<a href="https://colab.research.google.com/github/DharaCS23181/Skincare_Product_Prediction/blob/main/SVM(80_20).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import tensorflow as tf

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/ML/skin_recommendation_dataset.csv")

In [105]:
import re

def clean_text(x):
    # convert to string
    x = str(x)

    # remove everything except letters, numbers, space, dash, comma
    x = re.sub(r"[^a-zA-Z0-9,\-\s]", "", x)

    # split
    items = x.split(",")

    # clean each word
    items = [i.strip().lower() for i in items if i.strip() != ""]

    return items

In [106]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, f1_score

In [107]:
data = df[['skintype','skin_condition','notable_effects','product_type']].copy()

In [108]:
data['skintype'] = data['skintype'].apply(clean_text)
data['skin_condition'] = data['skin_condition'].apply(clean_text)
data['notable_effects'] = data['notable_effects'].apply(clean_text)

In [110]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb_skin = MultiLabelBinarizer()
mlb_condition = MultiLabelBinarizer()
mlb_effects = MultiLabelBinarizer()

skin_features = pd.DataFrame(
    mlb_skin.fit_transform(data['skintype']),
    columns=mlb_skin.classes_
)

condition_features = pd.DataFrame(
    mlb_condition.fit_transform(data['skin_condition']),
    columns=mlb_condition.classes_
)

effects_features = pd.DataFrame(
    mlb_effects.fit_transform(data['notable_effects']),
    columns=mlb_effects.classes_
)

X = pd.concat([skin_features, condition_features, effects_features], axis=1)

In [111]:
print("Skin:", mlb_skin.classes_)
print("Condition:", mlb_condition.classes_)
print("Effect:", mlb_effects.classes_)

Skin: ['combination' 'dry' 'normal' 'oily' 'sensitive']
Condition: ['acne' 'dry and dehydrated skin' 'dull skin' 'enlarged pores'
 'impaired skin barrier' 'oily skin' 'pigmentation' 'redness'
 'skin imbalance' 'sun damage' 'sun protection' 'wrinkles']
Effect: ['acne-free' 'acne-spot' 'anti-aging' 'balancing' 'black-spot'
 'brightening' 'hydrating' 'moisturizing' 'no-whitecast' 'oil-control'
 'pore-care' 'refreshing' 'skin-barrier' 'soothing' 'uv-protection']


In [112]:
X = pd.concat([
    skin_features,
    condition_features,
    effects_features
], axis=1)

In [113]:
le = LabelEncoder()
y = le.fit_transform(data['product_type'])

In [114]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [115]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [116]:
svm = SVC(
    kernel='rbf',   # best for your dataset
    C=1,
    gamma='scale'
)

svm.fit(X_train_scaled, y_train)

SVC(C=1)

In [117]:
y_pred = svm.predict(X_test_scaled)

In [118]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.5224489795918368


In [119]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.64      0.45      0.53        40
           1       0.41      0.40      0.40        50
           2       0.51      0.71      0.59        62
           3       0.73      0.64      0.68        42
           4       0.42      0.37      0.40        51

    accuracy                           0.52       245
   macro avg       0.54      0.52      0.52       245
weighted avg       0.53      0.52      0.52       245



In [120]:
f1 = f1_score(y_test, y_pred, average='weighted')

print("Weighted F1 Score:", f1)

Weighted F1 Score: 0.5189376180188329


In [121]:
print("Skin classes:", mlb_skin.classes_)
print("Condition classes:", mlb_condition.classes_)
print("Effect classes:", mlb_effects.classes_)

Skin classes: ['combination' 'dry' 'normal' 'oily' 'sensitive']
Condition classes: ['acne' 'dry and dehydrated skin' 'dull skin' 'enlarged pores'
 'impaired skin barrier' 'oily skin' 'pigmentation' 'redness'
 'skin imbalance' 'sun damage' 'sun protection' 'wrinkles']
Effect classes: ['acne-free' 'acne-spot' 'anti-aging' 'balancing' 'black-spot'
 'brightening' 'hydrating' 'moisturizing' 'no-whitecast' 'oil-control'
 'pore-care' 'refreshing' 'skin-barrier' 'soothing' 'uv-protection']


In [122]:
newdf.head()

,skintype,skin_condition,product_type,product_name,brand,notable_effects,picture_src
0,Oily,"('wrinkles', 'pigmentation', 'acne', 'enlarged...",Face Wash,ACWELL Bubble Free PH Balancing Cleanser,ACWELL,"('acne-free', 'pore-care', 'brightening', 'ant...",https://www.beautyhaul.com/assets/uploads/prod...
1,"Normal, Dry, Combination","('redness', 'skin imbalance')",Face Wash,ACWELL pH Balancing Soothing Cleansing Foam,ACWELL,"('soothing', 'balancing')",https://images.soco.id/8f08ced0-344d-41f4-a15e...
2,"Normal, Dry, Oily, Combination, Sensitive","('redness', 'skin imbalance')",Toner,Acwell Licorice pH Balancing Cleansing Toner,ACWELL,"('soothing', 'balancing')","https://www.soco.id/cdn-cgi/image/w=73,format=..."
3,Oily,"('wrinkles', 'pigmentation', 'acne', 'enlarged...",Toner,ACWELL Aquaseal Soothing Tonic,ACWELL,"('acne-free', 'pore-care', 'brightening', 'ant...",https://www.beautyhaul.com/assets/uploads/prod...
4,"Normal, Dry","('pigmentation', 'redness')",Toner,Licorice pH Balancing Essence Mist,ACWELL,"('brightening', 'soothing')","https://www.sociolla.com/cdn-cgi/image/w=425,f..."


In [123]:
user_skin = ['oily']
user_condition = ['acne']
user_effect = ['oil-control']

skin_vec = mlb_skin.transform([user_skin])
cond_vec = mlb_condition.transform([user_condition])
effect_vec = mlb_effects.transform([user_effect])

user_input = np.concatenate([skin_vec, cond_vec, effect_vec], axis=1)

# IMPORTANT: scale input
user_input_scaled = scaler.transform(user_input)

prediction = svm.predict(user_input_scaled)

predicted_type = le.inverse_transform(prediction)

print("Recommended Product Type:", predicted_type[0])

Recommended Product Type: Face Wash


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [133]:
import pandas as pd
import numpy as np

# ==============================
# 1. USER INPUT (NUMBER BASED)
# ==============================

print("\nSelect Skin Type:")
for i, val in enumerate(mlb_skin.classes_):
    print(i, ":", val)

skin_choice = int(input("Enter number: "))
user_skin = [mlb_skin.classes_[skin_choice]]


print("\nSelect Skin Condition:")
for i, val in enumerate(mlb_condition.classes_):
    print(i, ":", val)

cond_choice = int(input("Enter number: "))
user_condition = [mlb_condition.classes_[cond_choice]]


print("\nSelect Effect:")
for i, val in enumerate(mlb_effects.classes_):
    print(i, ":", val)

effect_choice = int(input("Enter number: "))
user_effect = [mlb_effects.classes_[effect_choice]]


# ==============================
# 2. CREATE INPUT VECTOR
# ==============================

user_df = pd.DataFrame(columns=X.columns)
user_df.loc[0] = 0

for val in user_skin + user_condition + user_effect:
    if val in user_df.columns:
        user_df.loc[0, val] = 1


# ==============================
# 3. SCALE INPUT
# ==============================

user_input_scaled = scaler.transform(user_df)


# ==============================
# 4. SVM PREDICTION
# ==============================

prediction = svm.predict(user_input_scaled)
predicted_type = le.inverse_transform(prediction)

print("\n✅ Predicted Product Type:", predicted_type[0])


# ==============================
# 5. SMART RECOMMENDATION SYSTEM
# ==============================

filtered = df[df['product_type'] == predicted_type[0]].copy()

# clean text columns
filtered['notable_effects'] = filtered['notable_effects'].astype(str).str.lower()
filtered['skin_condition'] = filtered['skin_condition'].astype(str).str.lower()
filtered['skintype'] = filtered['skintype'].astype(str).str.lower()


# scoring function
def calculate_score(row):
    score = 0

    # effect match (highest priority)
    if user_effect[0] in row['notable_effects']:
        score += 3

    # condition match
    if user_condition[0] in row['skin_condition']:
        score += 2

    # skin type match
    if user_skin[0] in row['skintype']:
        score += 1

    return score


# apply scoring
filtered['score'] = filtered.apply(calculate_score, axis=1)

# sort best matches
filtered = filtered.sort_values(by='score', ascending=False)


# ==============================
# 6. SHOW TOP 3 RESULTS
# ==============================

print("\n🔥 Top Recommended Products:\n")

top_results = filtered.head(3)

for i, row in top_results.iterrows():
    print("🔹 Product Name:", row['product_name'])
    print("🔹 Brand:", row['brand'])
    print("🔹 Effects:", ", ".join(clean_text(row['notable_effects'])))
    print("🔹 Score:", row['score'])
    print("🔹 Image URL:", row['picture_src'])
    print("-"*40)


Select Skin Type:
0 : combination
1 : dry
2 : normal
3 : oily
4 : sensitive
Enter number: 4

Select Skin Condition:
0 : acne
1 : dry and dehydrated skin
2 : dull skin
3 : enlarged pores
4 : impaired skin barrier
5 : oily skin
6 : pigmentation
7 : redness
8 : skin imbalance
9 : sun damage
10 : sun protection
11 : wrinkles
Enter number: 0

Select Effect:
0 : acne-free
1 : acne-spot
2 : anti-aging
3 : balancing
4 : black-spot
5 : brightening
6 : hydrating
7 : moisturizing
8 : no-whitecast
9 : oil-control
10 : pore-care
11 : refreshing
12 : skin-barrier
13 : soothing
14 : uv-protection
Enter number: 0

✅ Predicted Product Type: Face Wash

🔥 Top Recommended Products:

🔹 Product Name: CETAPHIL Oily Skin Cleanser 125ml
🔹 Brand: CETAPHIL
🔹 Effects: acne-free, oil-control, pore-care
🔹 Score: 6
🔹 Image URL: https://www.beautyhaul.com/assets/uploads/products/thumbs/800x800/jual_Oily_Skin_Cleanser_125ml_cetaphil.jpg
----------------------------------------
🔹 Product Name: BIO ESSENCE White Advanc